# Train the yard command adapter (LoRA)

Adapted from the LocoLabo capstone notebook by Michael Borck (Curtin, Semester 2 2026), for the yard voice-command practice project.

It takes `data/train_conversations.jsonl` and trains a LoRA adapter on **Qwen3-1.7B**, then exports a file Ollama can run.

## Before you start

- Run `python src/generate_data.py` on your laptop so `data/train_conversations.jsonl` exists (about 210 examples).
- Never train on `data/test.jsonl`. It is kept for testing.
- **Runtime → Change runtime type → T4 GPU.** The free tier is enough.

With this little data, training takes minutes. Installing, downloading the model and exporting take most of the time (about 30 to 60 minutes in total).


## Step 0: Confirm you have a GPU

If this errors or shows no GPU, fix the runtime type before going further.


In [1]:
!nvidia-smi


Fri Nov 20 15:24:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   59C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Step 1: Install Unsloth

Unsloth makes LoRA training fit in a free Colab GPU. This takes a few minutes and
may ask you to restart the session. If it does, restart and run from here.


In [2]:
!pip install -q unsloth unsloth_zoo


## Step 2: Settings

Everything you might want to change lives here.

The defaults are sensible starting points and match the other adapters in the
project, so your results stay comparable. Change them deliberately, one at a
time, and record what you changed and why in `TRAINING_LOG.md`.

| Setting | What it does |
|---|---|
| `LORA_R` | How much capacity the specialisation layer has. Higher learns more, overfits sooner. 8 to 32 is the useful range. |
| `EPOCHS` | How many times the model sees your whole dataset. Too many and it memorises rather than generalises. |
| `LEARNING_RATE` | How big a step each update takes. Lower it if the loss jumps around. |
| `MAX_SEQ_LENGTH` | Longest example, in tokens. Raise it if your answers are long, at the cost of memory. |


In [3]:
ADAPTER_NAME = "yard"   # lowercase, no spaces

BASE_MODEL = "unsloth/Qwen3-8B-unsloth-bnb-4bit"
MAX_SEQ_LENGTH = 512

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

LEARNING_RATE = 2e-4
EPOCHS = 3
BATCH_SIZE = 2
GRAD_ACCUM = 4          # effective batch size is BATCH_SIZE * GRAD_ACCUM

QUANT_METHOD = "q4_k_m"
SEED = 42               # leave this alone so your runs are reproducible


## Step 3: Load your training data

Run the cell and pick `train_conversations.jsonl` from the `data` folder on your laptop.


In [4]:
from google.colab import files

uploaded = files.upload()
DATA_PATH = next(iter(uploaded))
print(f"Using {DATA_PATH}")


Saving train_conversations.jsonl to train_conversations.jsonl
Using train_conversations.jsonl


## Step 4: Check the data before training on it

Worth its few seconds. A format error caught here costs nothing; the same error
caught after a three hour training run costs an afternoon.

This checks every line parses, has the `conversations` structure the trainer
expects, alternates user then assistant, and has no empty answers. It also flags
duplicates, which quietly waste training capacity.


In [5]:
import json
from collections import Counter

records, problems = [], []
with open(DATA_PATH) as f:
    for n, line in enumerate(f, 1):
        line = line.strip()
        if not line:
            continue
        try:
            rec = json.loads(line)
        except json.JSONDecodeError as e:
            problems.append(f"line {n}: not valid JSON ({e})")
            continue

        convo = rec.get("conversations")
        if not isinstance(convo, list) or len(convo) < 2:
            problems.append(f"line {n}: needs a 'conversations' list of at least 2 turns")
            continue
        roles = [t.get("role") for t in convo]
        if roles[0] != "user" or roles[1] != "assistant":
            problems.append(f"line {n}: first two turns must be user then assistant, got {roles[:2]}")
            continue
        if not str(convo[1].get("content", "")).strip():
            problems.append(f"line {n}: assistant answer is empty")
            continue
        records.append(rec)

questions = [r["conversations"][0].get("content", "") for r in records]
dupes = [q for q, c in Counter(questions).items() if c > 1]

print(f"Valid examples: {len(records)}")
print(f"Duplicate questions: {len(dupes)}")
if problems:
    print(f"\nProblems found: {len(problems)}")
    for p in problems[:10]:
        print("  " + p)
    if len(problems) > 10:
        print(f"  ... and {len(problems) - 10} more")

if len(records) < 150:
    print(f"\nNote: only {len(records)} examples. Run generate_data.py with a bigger --n.")

assert records, "No usable examples. Fix the format before continuing."


Valid examples: 210
Duplicate questions: 0


### Look at your data

Read a couple of examples with your own eyes. The model learns to imitate the
assistant answers, so if these are not good, nothing downstream will be.


In [6]:
for rec in records[:2]:
    turns = rec["conversations"]
    print("USER:     ", turns[0]["content"][:300])
    print("ASSISTANT:", turns[1]["content"][:300])
    print("-" * 70)


USER:      You turn a yard worker's spoken instruction into JSON.
Fields:
- action: "move", "park", "release" or "status", or null if unclear.
- unit: the trailer number as 4 digits (write numbers said in words as digits), or null.
- to: where it goes: "door_01" to "door_08" or "staging" for move, always "park
ASSISTANT: {"action": "move", "unit": "6188", "to": "door_08"}
----------------------------------------------------------------------
USER:      You turn a yard worker's spoken instruction into JSON.
Fields:
- action: "move", "park", "release" or "status", or null if unclear.
- unit: the trailer number as 4 digits (write numbers said in words as digits), or null.
- to: where it goes: "door_01" to "door_08" or "staging" for move, always "park
ASSISTANT: {"action": "move", "unit": "3815", "to": "door_07"}
----------------------------------------------------------------------


## Step 5: Load the base model

This is the model you are trying to beat. It loads in 4-bit precision, which is
what makes a 1.7 billion parameter model fit in a free Colab GPU.

First run downloads several GB, so it takes a few minutes.


In [7]:
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    full_finetuning=False,
)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1120 15:36:07.403000 567 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1120 15:36:07.456000 567 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130, so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.7: Fast Qwen3 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/144k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Unsloth: compiling each transformer block as one region (Qwen3DecoderLayer).


## Step 6: Add the specialisation layer

This is the LoRA part, and the heart of the whole approach.

Rather than retraining 1.7 billion parameters, which no free GPU could manage,
you freeze the base model and train a small set of extra weights alongside it.
The printout below shows how small: typically well under 1% of the total.

That is the claim this project is testing. A tiny number of well-chosen trained
parameters, on the right data, beating a much larger general model at one task.


In [8]:
model = FastModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100 * trainable / total:.3f}%)")


Trainable: 15,335,424 / 5,207,143,424 (0.295%)


## Step 7: Format the examples

Every model expects conversations laid out in its own particular way, with its
own markers for where a turn starts and ends. `apply_chat_template` handles that
so you do not have to.

The printed sample shows what the model actually sees. Worth a look once.


In [9]:
from datasets import Dataset

dataset = Dataset.from_list(records)

def format_conversations(examples):
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        for convo in examples["conversations"]
    ]
    return {"text": texts}

dataset = dataset.map(format_conversations, batched=True)
print(dataset[0]["text"][:600])


Map:   0%|          | 0/210 [00:00<?, ? examples/s]

<|im_start|>user
You turn a yard worker's spoken instruction into JSON.
Fields:
- action: "move", "park", "release" or "status", or null if unclear.
- unit: the trailer number as 4 digits (write numbers said in words as digits), or null.
- to: where it goes: "door_01" to "door_08" or "staging" for move, always "parking" for park, null for release and status or if no place was said.
If the worker corrects themselves, use the last thing they said.
Instruction: take 5032, no wait, 6188 to door eight<|im_end|>
<|im_start|>assistant
<think>

</think>

{"action": "move", "unit": "6188", "to": "door_


## Step 8: Train

**Watch the loss.** It should fall and then flatten out. If it falls to almost nothing, the model is memorising the examples rather than learning the pattern; try fewer epochs. If it bounces around without settling, lower the learning rate.

Write the final loss and the settings you used in your notes, even if the run fails.


In [11]:
from transformers import TrainingArguments
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=TrainingArguments(
        output_dir="checkpoints",
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_steps=8,
        fp16=True,
        logging_steps=5,
        save_strategy="epoch",
        seed=SEED,
        optim="adamw_8bit",
        report_to="none",
    ),
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
)

stats = trainer.train()
print(stats)


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/210 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 210 | Num Epochs = 3 | Total steps = 81
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 15,335,424 of 8,206,070,784 (0.19% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
5,3.733317
10,2.825575
15,1.835865
20,0.971084
25,0.395871
30,0.260901
35,0.245816
40,0.231045
45,0.212312
50,0.207194


Unsloth: Restored added_tokens_decoder metadata in checkpoints/checkpoint-27/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in checkpoints/checkpoint-54/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in checkpoints/checkpoint-81/tokenizer_config.json.


TrainOutput(global_step=81, training_loss=0.7470219941050918, metrics={'train_runtime': 508.0906, 'train_samples_per_second': 1.24, 'train_steps_per_second': 0.159, 'total_flos': 4810109023438848.0, 'train_loss': 0.7470219941050918, 'epoch': 3.0})


## Step 9: Export for Ollama

This merges your trained layer into the base model and compresses the result to
about 1 to 1.5 GB, which is the file Ollama runs.

Takes 10 to 20 minutes and is disk hungry. If Colab runs out of space, restart
the runtime and run everything except this cell again.


In [14]:
# Backup: save the small LoRA adapter before the export
# 1) to Colab's own disk (survives a crash, lost when the session ends)
model.save_pretrained("yard_lora_adapter")
tokenizer.save_pretrained("yard_lora_adapter")
!ls -lh yard_lora_adapter
# 2) to Google Drive (survives closing the tab)
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
!mkdir -p /content/drive/MyDrive/yard-voice-commands
!cp -r yard_lora_adapter /content/drive/MyDrive/yard-voice-commands/
!ls -lh /content/drive/MyDrive/yard-voice-commands/yard_lora_adapter

Unsloth: Restored added_tokens_decoder metadata in yard_lora_adapter/tokenizer_config.json.


total 70M
-rw-r--r-- 1 root root 1.3K Nov 20 16:00 adapter_config.json
-rw------- 1 root root  59M Nov 20 16:00 adapter_model.safetensors
-rw-r--r-- 1 root root 4.6K Nov 20 16:00 chat_template.jinja
-rw-r--r-- 1 root root 5.2K Nov 20 16:00 README.md
-rw-r--r-- 1 root root 5.1K Nov 20 16:00 tokenizer_config.json
-rw-r--r-- 1 root root  11M Nov 20 16:00 tokenizer.json


MessageError: Error: credential propagation was unsuccessful

In [ ]:
OUTPUT_DIR = f"{ADAPTER_NAME}_gguf"

model.save_pretrained_gguf(OUTPUT_DIR, tokenizer, quantization_method=QUANT_METHOD)

import os
for f in os.listdir(OUTPUT_DIR):
    if f.endswith(".gguf"):
        size_gb = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / (1024 ** 3)
        print(f"{f}  ({size_gb:.1f} GB)")


## Step 10: Get the file out of Colab

Copy it to Google Drive rather than downloading through the browser. **Do this before closing the tab**: Colab deletes everything when the session ends.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!mkdir -p "/content/drive/MyDrive/yard-voice-commands/{ADAPTER_NAME}"
!cp {OUTPUT_DIR}/*.gguf "/content/drive/MyDrive/yard-voice-commands/{ADAPTER_NAME}/"
!ls -lh "/content/drive/MyDrive/yard-voice-commands/{ADAPTER_NAME}/"


## What happens next (on your laptop)

1. Download the `.gguf` file from Google Drive into a folder, e.g. `models/` (keep it out of GitHub; it's too big).
2. Make a file called `Modelfile` next to it with one line (use your file's real name):

```
FROM ./yard.Q4_K_M.gguf
```

3. Register it with Ollama:

```
ollama create yard-lora -f Modelfile
ollama run yard-lora
```

4. Test it on `data/test.jsonl` with the **same prompt** the training used (`src/prompt.py`) and compare it with the base model `qwen3:1.7b` on the same test set.

If the score did not improve, that's a real result too. Read some of the wrong answers before concluding anything: check whether the labels are right, whether the base model was already good, and whether you trained too little or too much.
